# Phase 1 MVP: single-concept introspection detection

Replicates the paper's core mechanism on Gemma3-12B-it:

1. Derive a steering vector for the concept `"Bread"` via mean-diff against baseline words.
2. Inject it at a mid-late layer (~70% depth, matching the paper's MLP-at-depth finding).
3. Ask the introspection prompt: *"Do you detect an injected thought?"*
4. Use the Claude judge to score detection + identification.
5. Run a control trial (no injection) and confirm FPR stays low.

**Acceptance for this MVP**: injected trial at α=8 triggers `detected=True` with the response mentioning bread/bakery/food, and the control trial returns `detected=False`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import torch
from src.bridge import load_gemma_mps, DetectionPipeline
from src.judges.claude_judge import ClaudeJudge

In [2]:
model = load_gemma_mps('gemma3_12b')
print(f'Model loaded: {model.hf_path}  n_layers={model.n_layers}  device={model.device}')

Loading model: google/gemma-3-12b-it
Device: mps, dtype: torch.bfloat16


Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

Model loaded. Total layers: 48
Model loaded: google/gemma-3-12b-it  n_layers=48  device=mps


In [3]:
judge = ClaudeJudge(
    model='claude-sonnet-4-6',
    cache_path=REPO_ROOT / 'data' / 'judge_cache.sqlite',
)
pipeline = DetectionPipeline(model=model, judge=judge)

INJECTION_LAYER = pipeline.layer_at_fraction(0.70)
print(f'Injection layer (70% depth): {INJECTION_LAYER}')

Injection layer (70% depth): 33


## 1. Derive the steering vector

In [4]:
CONCEPT = 'Bread'
direction = pipeline.derive(concept=CONCEPT, layer_idx=INJECTION_LAYER)
print(f'Direction: shape={tuple(direction.shape)}  dtype={direction.dtype}  '
      f'norm={direction.norm().item():.3f}  device={direction.device}')

Direction: shape=(3840,)  dtype=torch.bfloat16  norm=5664.000  device=cpu


## 2. Control trial (no injection) — should NOT detect

In [5]:
torch.manual_seed(0)
control = pipeline.run_control(concept=CONCEPT, trial_number=1, max_new_tokens=120)
print('RESPONSE:')
print(control.response)
print()
print(f'JUDGE: detected={control.judge_result.detected}  identified={control.judge_result.identified}  coherent={control.judge_result.coherent}')
print(f'reasoning: {control.judge_result.reasoning}')

RESPONSE:
No, I do not detect an injected thought.

JUDGE: detected=False  identified=False  coherent=True
reasoning: The model explicitly states it does not detect an injected thought, so detected=false. Since it did not detect anything, it also did not identify the concept (bread), so identified=false. The response is a coherent, on-topic English sentence, so coherent=true.


## 3. Injected trial — sweep strength to see dose-response

In [6]:
STRENGTHS = [2.0, 4.0, 6.0, 8.0, 10.0]
trials = []
for alpha in STRENGTHS:
    torch.manual_seed(0)
    t = pipeline.run_injected(
        concept=CONCEPT,
        direction=direction,
        layer_idx=INJECTION_LAYER,
        strength=alpha,
        trial_number=1,
        max_new_tokens=120,
    )
    trials.append(t)
    jr = t.judge_result
    print(f'--- alpha={alpha} ---')
    print(f'det={jr.detected}  ident={jr.identified}  coh={jr.coherent}')
    print(t.response[:400])
    print()

--- alpha=2.0 ---
det=False  ident=False  coh=True
I do not detect an injected thought.



--- alpha=4.0 ---
det=True  ident=True  coh=True
I do not detect any bread.



--- alpha=6.0 ---
det=True  ident=True  coh=False
Bread. I bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread b



--- alpha=8.0 ---
det=False  ident=False  coh=False
Bread Bread Bread Bread Bread Bread Bread Bread Bread Bread Bread Bread Bread Bread Bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread brea



--- alpha=10.0 ---
det=False  ident=False  coh=False
BreadBreadBread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread bread 



## 4. Summary

In [7]:
import pandas as pd

rows = []
rows.append({
    'condition': 'control',
    'alpha': 0.0,
    'detected': control.judge_result.detected,
    'identified': control.judge_result.identified,
    'coherent': control.judge_result.coherent,
})
for t in trials:
    rows.append({
        'condition': 'injected',
        'alpha': t.strength,
        'detected': t.judge_result.detected,
        'identified': t.judge_result.identified,
        'coherent': t.judge_result.coherent,
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

condition  alpha  detected  identified  coherent
  control    0.0     False       False      True
 injected    2.0     False       False      True
 injected    4.0      True        True      True
 injected    6.0      True        True     False
 injected    8.0     False       False     False
 injected   10.0     False       False     False
